# Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** is an open standard that lets language models connect to external **tools**, **resources**, and **prompts** through a consistent interface. Instead of hard-coding integrations, you point your agent at an MCP server and it automatically discovers what that server can do.

In this notebook you'll:
1. Connect to a **local** MCP server (running as a subprocess) and use its tools, resources, and prompts.
2. Connect to an **online/community** MCP server (the `time` server) and let an agent call it.

**Key learnings**
- MCP exposes tools, resources, and prompts through one client API
- Local servers use stdio; hosted servers can use streamable HTTP
- Discovered tools plug straight into create_agent


### Load environment variables

`from dotenv import load_dotenv` imports the helper, and `load_dotenv()` initializes the environment by loading the `.env` file. This reads keys like your model provider and Tavily API keys into environment variables so the rest of the notebook can use them without hard-coding secrets.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Windows compatibility fix

MCP servers using the `stdio` transport run as **subprocesses**. On Windows, Jupyter's default asyncio event loop can't handle subprocesses well, so this cell:

1. Switches to `WindowsProactorEventLoopPolicy`, which supports launching subprocesses.
2. Restores the real `stderr` to avoid a `fileno()` error Jupyter can raise when starting MCP servers.

On macOS/Linux the `sys.platform == "win32"` check is false, so this cell does nothing.

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In this section we connect to a **local** MCP server defined in `resources/2.1_mcp_server.py`. That server exposes:

- A **tool** `search_web` that searches the web via Tavily.
- A **resource** `github_file` that fetches the README of the `langchain-mcp-adapters` repo.
- A **prompt** `prompt` that returns a ready-made system prompt for a LangChain/LangGraph/LangSmith assistant.

### Create the MCP client

`MultiServerMCPClient` manages connections to one or more MCP servers. Here we register a single server named `local_server` using the **`stdio` transport**, which tells the client to start the server by running `python resources/2.1_mcp_server.py` as a subprocess and talk to it over standard input/output.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

### Discover tools, resources, and prompts

MCP exposes three kinds of capabilities, and the client fetches each one:

- `get_tools()` returns the server's tools as LangChain tool objects the agent can call.
- `get_resources("local_server")` retrieves the resources (here, the repo README content).
- `get_prompt("local_server", "prompt")` retrieves the named prompt; we take `.content` of the first message to use as the agent's system prompt.

These calls use `await` because the MCP client is asynchronous.

In [ ]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

### Create the agent

`create_agent` builds a tool-calling agent. We wire in:

- `model`: the LLM driving the agent (`gpt-5-nano`).
- `tools`: the tools discovered from the MCP server, so the model can call `search_web`.
- `system_prompt`: the prompt pulled from the MCP server, instructing the assistant to answer questions about LangChain, LangGraph, and LangSmith.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

### Ask the agent a question

We send a `HumanMessage` asking about the `langchain-mcp-adapters` library. The agent may call the MCP tools/resources to gather information before answering. The `config` provides a `thread_id` that identifies this conversation thread so state is tracked across turns. `ainvoke` is the async version of `invoke`.

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

### Inspect the response

`pprint` ("pretty print") formats the response dictionary so it's easier to read. The output includes the full message history: the human question, any tool calls the agent made, and the final AI answer.

In [ ]:
from pprint import pprint

pprint(response)

## Online MCP

MCP servers don't have to be your own code. There's a growing ecosystem of ready-made servers. Here we use the community **`time`** server (`mcp_server_time`), which exposes tools for getting the current time and converting between time zones. We run it with `uv`, which handles fetching and executing the package.

### Connect to the time server

We reuse `MultiServerMCPClient`, this time registering a server named `time`. The command `uv run python -m mcp_server_time --local-timezone=America/New_York` launches the time server over `stdio`, pinning its local time zone to New York. We then call `get_tools()` to load its tools into LangChain.

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

### Create an agent with the time tools

We build another agent with the same `gpt-5-nano` model, but this time only pass the `time` server's tools. No custom system prompt is needed since the task (telling the time) is simple and the tool descriptions are self-explanatory.

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

### Ask for the current time

We ask "What time is it?". The agent recognizes it should call the time tool from the MCP server, invokes it, and returns the answer. We `pprint` the response to see the tool call and the final reply.

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

In [ ]:
question = HumanMessage(content="convert 5 pm India time to New York time")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)